# Chapter 4 - Nonparametric Filters: Particle Filter (Monte Carlo Localization)

The particle filter represents the belief as a set of weighted samples (particles).

    x_t^[i] ~ p(x_t | u_t, x_{t-1}^[i])    (sample from motion model)
    w_t^[i]  proportional to p(z_t | x_t^[i], m)  (weight by measurement likelihood)

Each particle is a hypothesis. Its weight is that hypothesis's right to survive
after seeing the measurement. When weight collapses, ESS drops and resampling is triggered.

Why EKF fails here: a symmetric environment creates a multimodal posterior.
A Gaussian cannot represent two modes simultaneously.

In [1]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pluto_filters.particle_filters.particle_filter import AugmentedMCL, Particle
from pluto_filters.kalman_filters.ekf import LANDMARKS, normalize_angle

np.random.seed(7)
TRUE_POSE = np.array([1.0, 0.0, 0.0])

def true_meas(landmark_id, pose=None):
    if pose is None:
        pose = TRUE_POSE
    lx, ly = LANDMARKS[landmark_id]
    dx, dy = lx - pose[0], ly - pose[1]
    r   = np.sqrt(dx**2 + dy**2) + np.random.normal(0, 0.2)
    phi = normalize_angle(np.arctan2(dy, dx) - pose[2]) + np.random.normal(0, 0.05)
    return r, phi

print("Setup complete")
print("Landmarks:", dict(LANDMARKS))

/home/thailuu/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Setup complete
Landmarks: {0: array([3., 0.]), 1: array([-3.,  2.]), 2: array([ 0., -4.]), 3: array([4., 4.]), 4: array([-4., -3.])}


In [2]:
mcl = AugmentedMCL(n_particles=500, map_bounds=(-5, 5, -5, 5))
snapshots = []

def snapshot(label):
    xs = [p.x for p in mcl.particles]
    ys = [p.y for p in mcl.particles]
    ws = [p.weight for p in mcl.particles]
    snapshots.append((list(xs), list(ys), list(ws), label))

snapshot('Prior (uniform)')

r0, phi0 = true_meas(0)
mcl.update(0, r0, phi0)
mcl.resample()
snapshot(f'After obs LM0  ESS={mcl.effective_n():.0f}')

mcl.predict(0.5, 0.0, 0.5)
snapshot('After move +0.5m')

r2, phi2 = true_meas(2)
mcl.update(2, r2, phi2)
mcl.resample()
snapshot(f'Converged  ESS={mcl.effective_n():.0f}')

x_est, y_est, theta_est = mcl.mean_pose()
print(f"Mean pose : ({x_est:.3f}, {y_est:.3f}, theta={np.degrees(theta_est):.1f} deg)")
print(f"True pose : ({TRUE_POSE[0]:.3f}, {TRUE_POSE[1]:.3f}, 0.0 deg)")
print(f"Position error: {np.sqrt((x_est-TRUE_POSE[0])**2+(y_est-TRUE_POSE[1])**2):.3f} m")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Particle Filter - Swarm of Hypotheses', fontsize=14)
for ax, (xs, ys, ws, label) in zip(axes.flat, snapshots):
    ws_arr  = np.array(ws)
    ws_norm = ws_arr / (ws_arr.max() + 1e-300)
    ax.scatter(xs, ys, c=ws_norm, cmap='Blues', s=8, alpha=0.7, vmin=0, vmax=1)
    for lm_id, (lx, ly) in LANDMARKS.items():
        ax.plot(lx, ly, 'r^', ms=8)
        ax.annotate(f'LM{lm_id}', (lx+0.1, ly+0.1), fontsize=7)
    ax.plot(*TRUE_POSE[:2], 'g*', ms=14, zorder=10, label='True position')
    ax.set_xlim(-5.5, 5.5); ax.set_ylim(-5.5, 5.5)
    ax.set_title(label); ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('ch04_particle_filter.png', dpi=120)
plt.show()

Mean pose : (1.131, -1.951, theta=58.9 deg)
True pose : (1.000, 0.000, 0.0 deg)
Position error: 1.956 m


/tmp/ipykernel_40121/2382387648.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resampling Schemes: Systematic vs Multinomial vs Stratified

Three standard resampling algorithms. All produce the same expected output
but have different variance properties. Systematic resampling is the most
commonly used in practice because it has the lowest variance.

In [3]:
def systematic_resample(particles, weights):
    N   = len(particles)
    pos = (np.arange(N) + np.random.uniform(0, 1)) / N
    cum = np.cumsum(weights); cum /= cum[-1]
    return [particles[i] for i in np.searchsorted(cum, pos)]

def multinomial_resample(particles, weights):
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()
    idx = np.random.choice(len(particles), len(particles), p=weights, replace=True)
    return [particles[i] for i in idx]

def stratified_resample(particles, weights):
    N   = len(particles)
    pos = (np.arange(N) + np.random.uniform(0, 1, N)) / N
    cum = np.cumsum(weights); cum /= cum[-1]
    return [particles[i] for i in np.searchsorted(cum, pos)]

N_TRIALS = 1000
N_P      = 200
true_val = 2.0
errors = {m: [] for m in ['systematic', 'multinomial', 'stratified']}
rng4 = np.random.default_rng(99)

for _ in range(N_TRIALS):
    xs_  = rng4.normal(true_val, 1.0, N_P)
    ws_  = rng4.dirichlet(np.ones(N_P))

    class FP:
        def __init__(self, x, w): self.x = x; self.weight = w

    pts = [FP(x, w) for x, w in zip(xs_, ws_)]
    for method, fn in [('systematic', systematic_resample),
                        ('multinomial', multinomial_resample),
                        ('stratified',  stratified_resample)]:
        rs  = fn(pts, ws_)
        est = np.mean([p.x for p in rs])
        errors[method].append((est - true_val)**2)

print("=== Resampling variance (lower = better) ===")
for method, errs in errors.items():
    print(f"  {method:12s}  variance = {np.mean(errs):.6f}")
print()
print("Systematic and stratified have lower variance than multinomial.")
print("Systematic uses a single random offset; stratified uses one per stratum.")

=== Resampling variance (lower = better) ===
  systematic    variance = 0.010291
  multinomial   variance = 0.014647
  stratified    variance = 0.011050

Systematic and stratified have lower variance than multinomial.
Systematic uses a single random offset; stratified uses one per stratum.


## Kidnapped Robot: Does Augmented MCL Recover?

After the filter converges, the robot is teleported to a new location.
Augmented MCL detects weight collapse (low w_fast/w_slow ratio) and injects
random particles to allow recovery from the wrong location.

In [4]:
np.random.seed(42)
mcl_kr = AugmentedMCL(n_particles=500, map_bounds=(-5, 5, -5, 5))

# Converge at TRUE_POSE
for lm in [0, 2, 3]:
    r_, p_ = true_meas(lm, TRUE_POSE)
    mcl_kr.update(lm, r_, p_)
    mcl_kr.resample()
mcl_kr.predict(0.5, 0.0, 0.5)

pre_kidnap = [(p.x, p.y, p.weight) for p in mcl_kr.particles]
ess_before = mcl_kr.effective_n()

# Kidnap to new location
KIDNAP = np.array([3.0, 3.0, 0.0])
ess_log = [ess_before]

for step in range(15):
    for lm in [0, 2]:
        r_, p_ = true_meas(lm, KIDNAP)
        mcl_kr.update(lm, r_, p_)
    mcl_kr.resample()
    mcl_kr.predict(0.3, 0.0, 0.5)
    ess_log.append(mcl_kr.effective_n())

post_kidnap = [(p.x, p.y, p.weight) for p in mcl_kr.particles]
x_kr, y_kr, _ = mcl_kr.mean_pose()
err_kr = np.sqrt((x_kr - KIDNAP[0])**2 + (y_kr - KIDNAP[1])**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Kidnapped Robot - Augmented MCL Recovery', fontsize=13)

ax = axes[0]
xs_, ys_, ws_ = zip(*pre_kidnap)
ax.scatter(xs_, ys_, c=np.array(ws_)/max(ws_), cmap='Blues', s=8, alpha=0.7)
ax.plot(*TRUE_POSE[:2], 'g*', ms=14, label='True (before kidnap)')
ax.plot(*KIDNAP[:2],    'r*', ms=14, label='Kidnap target')
for lm_id, (lx, ly) in LANDMARKS.items(): ax.plot(lx, ly, 'k^', ms=7)
ax.set_title('Before kidnap (converged)'); ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.legend(fontsize=7); ax.set_aspect('equal')

ax = axes[1]
xs_, ys_, ws_ = zip(*post_kidnap)
ax.scatter(xs_, ys_, c=np.array(ws_)/max(ws_), cmap='Reds', s=8, alpha=0.7)
ax.plot(*TRUE_POSE[:2], 'g*', ms=14, label='Old location')
ax.plot(*KIDNAP[:2],    'gv', ms=14, label='Kidnap target')
ax.plot(x_kr, y_kr, 'b+', ms=14, mew=3, label=f'Estimate (err={err_kr:.2f}m)')
for lm_id, (lx, ly) in LANDMARKS.items(): ax.plot(lx, ly, 'k^', ms=7)
ax.set_title('After 15 steps (recovering)'); ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.legend(fontsize=7); ax.set_aspect('equal')

ax = axes[2]
ax.plot(ess_log, 'purple', lw=2, marker='o', ms=5)
ax.axvline(0, color='red', ls='--', label='Kidnap moment')
ax.axhline(0.5 * 500, color='orange', ls=':', label='ESS = N/2 threshold')
ax.set_xlabel('Step after kidnap'); ax.set_ylabel('ESS')
ax.set_title('ESS drops at kidnap -> random particles injected')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ch04_kidnapped.png', dpi=120)
plt.show()
print(f"ESS before kidnap: {ess_before:.1f}")
print(f"Position error after recovery: {err_kr:.3f} m")

ESS before kidnap: 500.0
Position error after recovery: 2.552 m


/tmp/ipykernel_40121/913242424.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Compare, Break, Measure

In [5]:
print("=== Compare: N_particles vs position error ===")
for N in [25, 100, 500, 2000]:
    m = AugmentedMCL(n_particles=N, map_bounds=(-5,5,-5,5))
    np.random.seed(42)
    for lm in [0, 2, 3]:
        r_, p_ = true_meas(lm, TRUE_POSE)
        m.update(lm, r_, p_)
    m.resample()
    m.predict(0.5, 0.0, 0.5)
    r_, p_ = true_meas(2, TRUE_POSE)
    m.update(2, r_, p_)
    m.resample()
    xm, ym, _ = m.mean_pose()
    err = np.sqrt((xm-TRUE_POSE[0])**2+(ym-TRUE_POSE[1])**2)
    print(f"  N={N:5d}: error={err:.3f}m  ESS={m.effective_n():.1f}")

print("\n=== Break: N=10 particles - particle deprivation ===")
m_tiny = AugmentedMCL(n_particles=10, map_bounds=(-5,5,-5,5))
for step in range(8):
    np.random.seed(step)
    r_, p_ = true_meas(0, TRUE_POSE)
    m_tiny.update(0, r_, p_)
    m_tiny.resample()
    nonzero = sum(p.weight > 1e-10 for p in m_tiny.particles)
    print(f"  step {step}: ESS={m_tiny.effective_n():.1f}  particles with weight > 0: {nonzero}")

print("\n=== Measure: resampling variance comparison ===")
for method, var_val in errors.items():
    print(f"  {method:12s}  mean squared error = {np.mean(var_val):.6f}")

=== Compare: N_particles vs position error ===
  N=   25: error=1.391m  ESS=25.0
  N=  100: error=0.749m  ESS=100.0
  N=  500: error=0.957m  ESS=500.0
  N= 2000: error=0.193m  ESS=2000.0

=== Break: N=10 particles - particle deprivation ===
  step 0: ESS=10.0  particles with weight > 0: 10
  step 1: ESS=10.0  particles with weight > 0: 10
  step 2: ESS=10.0  particles with weight > 0: 10
  step 3: ESS=10.0  particles with weight > 0: 10
  step 4: ESS=10.0  particles with weight > 0: 10
  step 5: ESS=10.0  particles with weight > 0: 10
  step 6: ESS=10.0  particles with weight > 0: 10
  step 7: ESS=10.0  particles with weight > 0: 10

=== Measure: resampling variance comparison ===
  systematic    mean squared error = 0.010291
  multinomial   mean squared error = 0.014647
  stratified    mean squared error = 0.011050
